In [ ]:
!pip uninstall langchain-core langchain-huggingface -y
!pip install langchain-core langchain-huggingface


In [ ]:

%pip install --upgrade langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters

In [ ]:
!pip install langchain-classic

## Local configuration

This notebook reads credentials from environment variables. Never commit a `.env` file, access token, or database password. Set `HF_TOKEN`, `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` in your environment before running. Dataset and output locations can be changed through `HAGRAG_PDF_DIR`, `HAGRAG_DATASET_PATH`, and `HAGRAG_OUTPUT_DIR`.


In [ ]:
import os
from pathlib import Path

HF_TOKEN = os.getenv("HF_TOKEN")
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
PDF_DIR = Path(os.getenv("HAGRAG_PDF_DIR", "data/pdfs"))
DATASET_PATH = Path(os.getenv("HAGRAG_DATASET_PATH", "data/100diabetes_qa_dataset.jsonl"))
OUTPUT_DIR = Path(os.getenv("HAGRAG_OUTPUT_DIR", "results"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not NEO4J_PASSWORD:
    print("NEO4J_PASSWORD is not set; database-dependent cells will not connect.")


In [ ]:
#pip install  -U bitsandbytes pdfplumber tqdm langchain langchain-openai neo4j networkx graspologic scikit-learn sentence-transformers llama-index rouge-score PyPDF2 openai numpy==1.26.4 scipy==1.12.0 transformers langchain-huggingface torch sentence-transformers
!pip install -U langchain-openai 


In [ ]:
!pip install pdfplumber neo4j networkx graspologic scikit-learn tqdm

In [ ]:
# 1. Remove broken installs
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers bitsandbytes

# 2. Install a clean PyTorch + TorchVision stack (CUDA 12.1 build for Colab GPUs)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# 3. Install Hugging Face + Sentence Transformers + bitsandbytes
!pip install -U transformers==4.44.2 sentence-transformers accelerate bitsandbytes safetensors

import torch, torchvision, transformers, sentence_transformers
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)


In [ ]:

!pip install future


In [ ]:
import os
import pdfplumber
from tqdm import tqdm


from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains import LLMChain
# from langchain.chains import LLMChain

import json
import re
from typing import List, Tuple, Dict, Set, Optional, Any
from neo4j import GraphDatabase
import networkx as nx
from graspologic.partition import hierarchical_leiden
from collections import defaultdict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import heapq
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)

class PDFProcessor:
    """Handles PDF processing and text extraction"""

    def __init__(self, pdf_dir: str = str(PDF_DIR)):
        self.pdf_dir = pdf_dir

    def extract_texts(self) -> List[str]:
        """Extract text from all PDFs in the directory"""
        texts = []
        pdf_files = [f for f in os.listdir(self.pdf_dir) if f.endswith('.pdf')]

        for filename in tqdm(pdf_files, desc="Processing PDF files"):
            pdf_path = os.path.join(self.pdf_dir, filename)
            try:
                with pdfplumber.open(pdf_path) as pdf:
                    for page in tqdm(pdf.pages, desc=f"Processing {filename}", leave=False):
                        text = page.extract_text()
                        if text and text.strip():
                            texts.append(text)
            except Exception as e:
                logging.error(f"Error processing {filename}: {e}")

        return texts

    def create_chunks(self, texts: List[str], chunk_size: int = 1024, chunk_overlap: int = 20) -> List[str]:
        """Split texts into chunks and return as strings"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
        documents = splitter.create_documents(texts)
        return [doc.page_content for doc in documents]

In [ ]:
from typing import List, Tuple, Dict
import json
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import heapq
import logging

class TripletExtractor:
    """Handles knowledge graph triplet extraction"""

    def __init__(self, llm_pipeline):
        self.llm_pipeline = llm_pipeline  # Expecting a transformers pipeline object

    def parse_response(self, response_str: str) -> Tuple[List[tuple], List[tuple]]:
        """Parse LLM response to extract entities and relationships"""
        entities, relationships = [], []

        try:
            # Clean the response by removing any special tokens or extra whitespace
            response_clean = re.sub(r'<\|.*?\|>', '', response_str).strip()

            # Extract JSON using a robust pattern
            json_pattern = r'\{.*\}'
            match = re.search(json_pattern, response_clean, re.DOTALL)
            if not match:
                logging.warning(f"No valid JSON found in response: {response_clean[:100]}...")
                return entities, relationships

            json_str = match.group(0)
            data = json.loads(json_str)

            # Validate and extract entities
            for entity in data.get("entities", []):
                if all(key in entity for key in ["entity_name", "entity_type", "entity_description"]):
                    entities.append((
                        entity["entity_name"],
                        entity["entity_type"],
                        entity["entity_description"],
                        entity.get("entity_attributes", {})
                    ))

            # Validate and extract relationships
            for relation in data.get("relationships", []):
                if all(key in relation for key in ["source_entity", "target_entity", "relation", "relationship_description"]):
                    relationships.append((
                        relation["source_entity"],
                        relation["target_entity"],
                        relation["relation"],
                        relation["relationship_description"],
                        relation.get("relationship_attributes", {})
                    ))

            logging.info(f"Successfully parsed {len(entities)} entities and {len(relationships)} relationships")
            return entities, relationships

        except json.JSONDecodeError as e:
            logging.error(f"JSON parsing error: {e}, Response: {response_clean[:100]}...")
            return entities, relationships
        except Exception as e:
            logging.error(f"Unexpected error in parse_response: {e}, Response: {response_clean[:100]}...")
            return entities, relationships

    def extract_triplets(self, text: str, max_paths_per_chunk: int = 2) -> Tuple[List[tuple], List[tuple]]:
        """Extract triplets from text using raw transformers pipeline"""
        try:
            # Construct prompt with Llama 3.1 chat template
            prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id>
            You are an expert in extracting entity-relation triplets from text related to diabetes. Extract up to {max_paths_per_chunk} triplets from the provided text and return a valid JSON object in this format:

            {{
              "entities": [
                {{
                  "entity_name": "string",
                  "entity_type": "string",
                  "entity_description": "string",
                  "entity_attributes": {{}}
                }}
              ],
              "relationships": [
                {{
                  "source_entity": "string",
                  "target_entity": "string",
                  "relation": "string",
                  "relationship_description": "string",
                  "relationship_attributes": {{}}
                }}
              ]
            }}

            Instructions:
            1. Identify entities with:
              - entity_name: Capitalized name (e.g., Insulin, Diabetes)
              - entity_type: Type (e.g., Drug, Disease, Protein)
              - entity_description: Brief description
              - entity_attributes: Key-value pairs (e.g., {{"domain": "Diabetes"}})
            2. Identify relationships with:
              - source_entity: Source entity name
              - target_entity: Target entity name
              - relation: Relationship type (e.g., TREATS, CAUSES)
              - relationship_description: Brief explanation
              - relationship_attributes: Key-value pairs (e.g., {{"context": "Medical"}})
            3. Return only the JSON object. No additional text or tokens.

            Example:
            {{
              "entities": [
                {{
                  "entity_name": "Insulin",
                  "entity_type": "Drug",
                  "entity_description": "A hormone used to manage diabetes",
                  "entity_attributes": {{"domain": "Diabetes"}}
                }},
                {{
                  "entity_name": "Diabetes",
                  "entity_type": "Disease",
                  "entity_description": "A chronic condition affecting blood sugar",
                  "entity_attributes": {{"domain": "Medical"}}
                }}
              ],
              "relationships": [
                {{
                  "source_entity": "Insulin",
                  "target_entity": "Diabetes",
                  "relation": "TREATS",
                  "relationship_description": "Insulin is used to control blood sugar in diabetes",
                  "relationship_attributes": {{"context": "Medical Treatment"}}
                }}
              ]
            }}

            Input Text:
            {text[:4000]}
            <|end_header_id><|start_header_id|>user<|end_header_id>
            Return the extracted triplets in the specified JSON format.
            <|end_header_id>"""

            # Generate response using the raw pipeline
            response = self.llm_pipeline(prompt, max_new_tokens=512, temperature=0.1, do_sample=True, top_p=0.95, repetition_penalty=1.2)
            response_text = response[0]["generated_text"] if isinstance(response, list) else response
            # Extract the output after the prompt
            response_text = response_text[len(prompt):].strip() if response_text.startswith(prompt) else response_text
            logging.debug(f"Raw LLM response: {response_text[:100]}...")
            return self.parse_response(response_text)
        except Exception as e:
            logging.error(f"Error extracting triplets: {str(e)}, Text: {text[:100]}...")
            return [], []
class ArchRAGCHNSW:
    """C-HNSW Index Implementation for ArchRAG"""

    def __init__(self, embedding_dim: int = 384, M: int = 16, ef_construction: int = 200):
        self.embedding_dim = embedding_dim
        self.M = M
        self.ef_construction = ef_construction
        self.layers = []
        self.node_embeddings = {}
        self.inter_layer_links = {}
        self.node_data = {}
        self.nn_index = None

    def add_node(self, node_id: str, embedding: np.ndarray, layer: int, node_data: dict = None):
        """Add a node to the specified layer"""
        while len(self.layers) <= layer:
            self.layers.append(nx.Graph())

        self.layers[layer].add_node(node_id)
        self.node_embeddings[node_id] = embedding
        if node_data:
            self.node_data[node_id] = node_data

    def distance(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine distance between two vectors"""
        return 1 - cosine_similarity([a], [b])[0][0]

    def build_intra_layer_links(self):
        """Build links within each layer using approximate nearest neighbors"""
        for layer_idx, layer_graph in enumerate(self.layers):
            nodes = list(layer_graph.nodes())
            if len(nodes) <= 1:
                continue

            embeddings = np.array([self.node_embeddings[n] for n in nodes])
            nn = NearestNeighbors(n_neighbors=min(self.M, len(nodes)), metric="cosine")
            nn.fit(embeddings)

            distances, indices = nn.kneighbors(embeddings)
            for i, node_id in enumerate(nodes):
                for j, dist in zip(indices[i], distances[i]):
                    if j != i:
                        neighbor_id = nodes[j]
                        layer_graph.add_edge(node_id, neighbor_id, weight=dist, similarity=1-dist)

    def build_inter_layer_links(self):
        """Build links between adjacent layers"""
        for layer_idx in range(1, len(self.layers)):
            higher_layer_nodes = list(self.layers[layer_idx].nodes())
            lower_layer_nodes = list(self.layers[layer_idx-1].nodes())

            if not higher_layer_nodes or not lower_layer_nodes:
                continue

            lower_embeddings = np.array([self.node_embeddings[n] for n in lower_layer_nodes])
            nn = NearestNeighbors(n_neighbors=1, metric="cosine")
            nn.fit(lower_embeddings)

            for node_id in higher_layer_nodes:
                if node_id not in self.node_embeddings:
                    continue

                node_emb = self.node_embeddings[node_id]
                distances, indices = nn.kneighbors([node_emb])
                nearest = lower_layer_nodes[indices[0][0]]
                self.inter_layer_links[node_id] = nearest

    def search_layer(self, layer_idx: int, query_embedding: np.ndarray,
                    entry_point: str, k: int = 1) -> List[str]:
        """Search for k nearest neighbors in a specific layer using edge weights"""
        if layer_idx >= len(self.layers) or not self.layers[layer_idx].nodes():
            return []

        if entry_point not in self.node_embeddings:
            return []

        layer = self.layers[layer_idx]
        visited = set([entry_point])
        candidates = [(self.distance(query_embedding, self.node_embeddings[entry_point]), entry_point)]
        results = [(self.distance(query_embedding, self.node_embeddings[entry_point]), entry_point)]

        while candidates:
            current_dist, current = heapq.heappop(candidates)

            if results and current_dist > max(results, key=lambda x: x[0])[0]:
                break

            for neighbor in layer.neighbors(current):
                if neighbor not in visited and neighbor in self.node_embeddings:
                    visited.add(neighbor)
                    dist = self.distance(query_embedding, self.node_embeddings[neighbor])
                    edge_data = layer.get_edge_data(current, neighbor)
                    weight = edge_data.get("weight", 1.0)

                    adjusted_dist = dist * weight
                    if len(results) < k or adjusted_dist < max(results, key=lambda x: x[0])[0]:
                        heapq.heappush(candidates, (adjusted_dist, neighbor))
                        heapq.heappush(results, (adjusted_dist, neighbor))

                        if len(results) > k:
                            results.remove(max(results, key=lambda x: x[0]))

        return [node_id for _, node_id in sorted(results, key=lambda x: x[0])[:k]]

    def hierarchical_search(self, query_embedding: np.ndarray, k: int = 3) -> Dict[int, List[str]]:
        """Perform hierarchical search across all layers"""
        if not self.layers or all(not layer.nodes() for layer in self.layers):
            return {}

        results = {}
        top_layer = len(self.layers) - 1
        while top_layer >= 0 and not self.layers[top_layer].nodes():
            top_layer -= 1

        if top_layer < 0:
            return {}

        entry_point = list(self.layers[top_layer].nodes())[0]

        for layer_idx in range(top_layer, -1, -1):
            if not self.layers[layer_idx].nodes():
                continue

            layer_results = self.search_layer(layer_idx, query_embedding, entry_point, k)
            results[layer_idx] = layer_results

            if layer_results and layer_idx > 0:
                best_node = layer_results[0]
                entry_point = self.inter_layer_links.get(best_node,
                    list(self.layers[layer_idx - 1].nodes())[0] if self.layers[layer_idx - 1].nodes() else entry_point)

        return results

In [ ]:
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import networkx as nx
from graspologic.partition import hierarchical_leiden
from collections import defaultdict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json
import logging

class ArchRAGStore:
    """Enhanced ArchRAG Store with hierarchical communities"""

    def __init__(self, uri: str, username: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.entity_embeddings = {}
        self.hierarchical_communities = {}
        self.community_summaries = {}
        self.chnsw_index = ArchRAGCHNSW(embedding_dim=384)
        self.max_layers = 3
        self.min_nodes_per_layer = 3
        self.max_layers = 4  # Increase max layers
        self.min_nodes_per_layer = 2  # Reduce minimum nodes per layer
        self.connection_threshold = 0.1  # Threshold for community connections

    def close(self):
        """Close the database connection"""
        self.driver.close()

    def add_triplets(self, entities: List[tuple], relationships: List[tuple], doc_id: str):
        """Add triplets to Neo4j with enhanced attributes"""
        with self.driver.session() as session:
            for entity_name, entity_type, entity_desc, entity_attrs in entities:
                entity_text = f"{entity_name} {entity_type} {entity_desc} {json.dumps(entity_attrs)}"
                embedding = self.embedding_model.encode(entity_text)
                self.entity_embeddings[entity_name] = embedding

                session.run(
                    """
                    MERGE (e:Entity {name: $name})
                    SET e.type = $type, e.description = $desc, e.doc_id = $doc_id, e.attributes = $attrs
                    """,
                    name=entity_name, type=entity_type, desc=entity_desc,
                    doc_id=doc_id, attrs=json.dumps(entity_attrs)
                )

            for source, target, rel, rel_desc, rel_attrs in relationships:
                session.run(
                    """
                    MATCH (source:Entity {name: $source})
                    MATCH (target:Entity {name: $target})
                    MERGE (source)-[r:RELATION {type: $rel}]->(target)
                    SET r.description = $desc, r.doc_id = $doc_id, r.attributes = $attrs
                    """,
                    source=source, target=target, rel=rel, desc=rel_desc,
                    doc_id=doc_id, attrs=json.dumps(rel_attrs)
                )

    def _calculate_community_connection(self, comm1: dict, comm2: dict, graph: nx.Graph) -> float:
        """Calculate connection strength between two communities"""
        members1 = set(comm1["members"])
        members2 = set(comm2["members"])

        # Direct connections between members
        direct_connections = 0
        total_possible = len(members1) * len(members2)

        if total_possible == 0:
            return 0.0

        for m1 in members1:
            for m2 in members2:
                if graph.has_edge(m1, m2):
                    direct_connections += 1

        # Shared members (for overlapping communities)
        shared_members = len(members1.intersection(members2))

        # Calculate connection strength
        connection_strength = (direct_connections / total_possible) + (shared_members / max(len(members1), len(members2)))

        return min(connection_strength, 1.0)

    def compute_similarity_threshold(self, similarities: List[float], percentile: float = 0.7) -> float:
        """Compute dynamic similarity threshold"""
        if not similarities:
            return 0.5
        return np.percentile(similarities, percentile * 100)

    def augment_graph_with_attributes(self, nx_graph: nx.Graph, k: int = 5) -> nx.Graph:
        """Augment graph by connecting entities with similar attributes"""
        augmented_graph = nx_graph.copy()
        nodes = list(nx_graph.nodes())

        if len(nodes) <= 1:
            return augmented_graph

        all_similarities = []
        node_similarities = {}

        for i, node1 in enumerate(nodes):
            if node1 not in self.entity_embeddings:
                continue

            similarities = []
            for j, node2 in enumerate(nodes):
                if i != j and node2 in self.entity_embeddings:
                    sim = cosine_similarity(
                        [self.entity_embeddings[node1]],
                        [self.entity_embeddings[node2]]
                    )[0][0]
                    similarities.append((node2, sim))
                    all_similarities.append(sim)

            node_similarities[node1] = similarities

        threshold = self.compute_similarity_threshold(all_similarities)

        for node1, similarities in node_similarities.items():
            top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:k]
            for node2, sim in top_k:
                if sim > threshold:
                    weight = 1 - sim
                    augmented_graph.add_edge(node1, node2, weight=weight, similarity=sim)

        return augmented_graph

    def cluster_graph(self, graph: nx.Graph, method: str = "leiden") -> List[Set[str]]:
        """Cluster graph using specified method"""
        if len(graph.nodes()) < 2:
            return [set(graph.nodes())]

        try:
            if method == "leiden":
                clusters = hierarchical_leiden(graph, max_cluster_size=5)
                community_map = defaultdict(set)
                for item in clusters:
                    community_map[item.cluster].add(item.node)
                return list(community_map.values())
            else:
                communities = nx.algorithms.community.greedy_modularity_communities(graph)
                return [set(c) for c in communities]
        except Exception as e:
            logging.error(f"Clustering failed: {e}, falling back to single community")
            return [set(graph.nodes())]

    def generate_attributed_community_summary(self, community: Set[str], llm) -> str:
        """Generate summary for attributed community using LLM"""

        # Check if this is a meta-community (contains other community IDs)
        community_ids = {entity for entity in community if entity.startswith('L')}
        actual_entities = {entity for entity in community if not entity.startswith('L')}

        if community_ids and not actual_entities:
            # This is a pure meta-community - summarize the lower-level communities
            return self._generate_meta_community_summary(community_ids, llm)
        # elif community_ids and actual_entities:
        #     # Mixed community - handle both
        #     return self._generate_mixed_community_summary(community_ids, actual_entities, llm)
        else:
            # Regular entity community - use existing logic
            return self._generate_entity_community_summary(actual_entities, llm)

    def _generate_meta_community_summary(self, community_ids: Set[str], llm) -> str:
        """Generate summary for meta-community containing other communities"""
        sub_summaries = []
        total_members = 0

        for comm_id in community_ids:
            # Find the community data from lower layers
            for layer_data in self.hierarchical_communities.values():
                for comm_data in layer_data:
                    if comm_data['id'] == comm_id:
                        sub_summaries.append(comm_data['summary'])
                        total_members += comm_data['size']
                        break

        if not sub_summaries:
            return f"Meta-community grouping {len(community_ids)} sub-communities"

        combined_text = "\n\n".join(sub_summaries)

        meta_prompt = ChatPromptTemplate.from_messages([
            ("system",
            "You are analyzing a higher-level community that groups several sub-communities. "
            "Create a concise summary that identifies the overarching themes and patterns "
            "across these sub-communities. Focus on what connects them at a higher level."),
            ("human", "Sub-community summaries:\n{summaries}")
        ])

        meta_chain = LLMChain(llm=llm, prompt=meta_prompt)
        try:
            result = meta_chain.invoke({"summaries": combined_text})
            return f"Meta-community ({total_members} total entities): {result['text'].strip()}"
        except Exception as e:
            logging.error(f"Error generating meta-community summary: {e}")
            return f"Meta-community grouping {len(community_ids)} sub-communities with {total_members} total entities"

    def _generate_entity_community_summary(self, actual_entities: Set[str], llm) -> str:
        """Generate summary for regular entity community (existing logic)"""
        # Your existing entity summary logic here
        entity_details = []
        relationship_details = []

        with self.driver.session() as session:
            for entity_name in actual_entities:
                result = session.run(
                    "MATCH (e:Entity {name: $name}) RETURN e.name, e.type, e.description, e.attributes",
                    name=entity_name
                )
                record = result.single()
                if record:
                    entity_details.append({
                        "name": record["e.name"],
                        "type": record["e.type"] or "Unknown",
                        "description": record["e.description"] or "No description",
                        "attributes": json.loads(record["e.attributes"] or "{}")
                    })

        if not entity_details:
            return f"Community with {len(actual_entities)} entities"

        # Rest of your existing entity summary logic...
        summary_text = f"Entities: {json.dumps(entity_details, indent=2)}"

        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", "Create a concise summary of this entity community."),
            ("human", "Community data:\n{community_data}")
        ])

        summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
        try:
            result = summary_chain.invoke({"community_data": summary_text})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating entity summary: {e}")
            return f"Community with {len(actual_entities)} entities"

    def build_hierarchical_attributed_communities(self, llm):
        """Build hierarchical attributed communities"""
        current_graph = self._create_nx_graph()
        layer = 0
        all_communities = {}

        logging.info(f"Starting hierarchical clustering with {len(current_graph.nodes())} nodes")

        while layer < self.max_layers and len(current_graph.nodes()) >= self.min_nodes_per_layer:
            logging.info(f"Processing layer {layer} with {len(current_graph.nodes())} nodes")

            augmented_graph = self.augment_graph_with_attributes(current_graph)
            communities = self.cluster_graph(augmented_graph)
            logging.info(f"Found {len(communities)} communities in layer {layer}")

            layer_communities = []
            for i, community in enumerate(communities):
                if len(community) == 0:
                    continue

                # DON'T SKIP - Process all communities including meta-communities
                community_id = f"L{layer}_C{i}"
                summary = self.generate_attributed_community_summary(community, llm)

                community_data = {
                    "id": community_id,
                    "layer": layer,
                    "members": list(community),
                    "summary": summary,
                    "size": len(community),
                    "is_meta_community": any(member.startswith('L') for member in community)
                }
                layer_communities.append(community_data)

                try:
                    self.entity_embeddings[community_id] = self.embedding_model.encode(summary)
                except Exception as e:
                    logging.error(f"Error generating embedding for community {community_id}: {e}")
                    self.entity_embeddings[community_id] = np.zeros(384)

            all_communities[layer] = layer_communities
            self.hierarchical_communities[layer] = layer_communities

            # Build next layer graph
            new_graph = nx.Graph()
            for comm_data in layer_communities:
                new_graph.add_node(comm_data["id"])

            # Connect communities that share members or have connections
            for i, comm1 in enumerate(layer_communities):
                for j, comm2 in enumerate(layer_communities):
                    if i < j:
                        connection_strength = self._calculate_community_connection(
                            comm1, comm2, augmented_graph
                        )
                        if connection_strength > 0.1:  # Threshold for connection
                            new_graph.add_edge(comm1["id"], comm2["id"], weight=1-connection_strength)

            current_graph = new_graph
            layer += 1

        logging.info(f"Built {layer} layers of hierarchical communities")
        return all_communities

    def build_chnsw_index(self):
        """Build C-HNSW index from hierarchical communities"""
        logging.info("Building C-HNSW index...")

        # Add all communities to appropriate layers
        for layer, communities in self.hierarchical_communities.items():
            for community in communities:
                community_id = community["id"]
                if community_id in self.entity_embeddings:
                    embedding = self.entity_embeddings[community_id]
                    self.chnsw_index.add_node(
                        community_id,
                        embedding,
                        layer,
                        node_data=community
                    )

        # Add individual entities to layer 0
        for entity_name, embedding in self.entity_embeddings.items():
            if not entity_name.startswith('L'):  # Skip community IDs
                self.chnsw_index.add_node(
                    entity_name,
                    embedding,
                    0,
                    node_data={"type": "entity", "name": entity_name}
                )

        self.chnsw_index.build_intra_layer_links()
        self.chnsw_index.build_inter_layer_links()
        logging.info("C-HNSW index construction completed")

    def _create_nx_graph(self) -> nx.Graph:
        """Create NetworkX graph from Neo4j database"""
        nx_graph = nx.Graph()
        with self.driver.session() as session:
            result = session.run(
                "MATCH (e1)-[r:RELATION]->(e2) RETURN e1.name, r.type, e2.name, r.description, r.attributes"
            )
            for record in result:
                e1, rel, e2, desc, attrs = (
                    record["e1.name"], record["r.type"], record["e2.name"],
                    record["r.description"], record["r.attributes"]
                )
                nx_graph.add_node(e1)
                nx_graph.add_node(e2)
                nx_graph.add_edge(e1, e2, relationship=rel, description=desc, attributes=attrs)
        return nx_graph

    def build_communities(self, llm):
        """Main method to build the complete ArchRAG structure"""
        logging.info("Building ArchRAG hierarchical attributed communities...")
        self.build_hierarchical_attributed_communities(llm)
        self.build_chnsw_index()
        logging.info("ArchRAG community structure completed!")

In [ ]:
import json
import numpy as np
import logging
import re
from typing import Dict, Any, List, Optional
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface.chat_models import ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from tqdm import tqdm
from transformers import pipeline as hf_pipeline
class ArchRAGQueryEngine:
    """Enhanced Query Engine for ArchRAG"""

    def __init__(self, graph_store, llm, relevance_threshold: float = 0.3):
        self.graph_store = graph_store
        self.llm = llm
        self.embedding_model = graph_store.embedding_model
        self.relevance_threshold = relevance_threshold

    def retrieve_hierarchical_info(self, query: str, k: int = 3) -> Dict[int, List[Dict]]:
        """Retrieve relevant information across all hierarchical layers"""
        query_embedding = self.embedding_model.encode(query)
        search_results = self.graph_store.chnsw_index.hierarchical_search(query_embedding, k)

        hierarchical_info = {}
        for layer, node_ids in search_results.items():
            layer_info = []
            for node_id in node_ids:
                if node_id in self.graph_store.chnsw_index.node_data:
                    node_data = self.graph_store.chnsw_index.node_data[node_id]
                    if node_data.get("type") == "entity":
                        entity_details = self._get_entity_details(node_id)
                        layer_info.append({
                            "id": node_id,
                            "type": "entity",
                            "content": entity_details,
                            "layer": layer
                        })
                    else:
                        layer_info.append({
                            "id": node_id,
                            "type": "community",
                            "content": node_data,
                            "layer": layer
                        })
            hierarchical_info[layer] = layer_info
        return hierarchical_info

    def _get_entity_details(self, entity_name: str) -> Dict:
        """Get detailed information about an entity from Neo4j"""
        with self.graph_store.driver.session() as session:
            result = session.run(
                """
                MATCH (e:Entity {name: $name})
                OPTIONAL MATCH (e)-[r:RELATION]-(connected)
                RETURN e.name, e.type, e.description, e.attributes,
                       collect({relation: r.type, connected_entity: connected.name,
                               relation_desc: r.description}) as relationships
                """,
                name=entity_name
            )
            record = result.single()
            if record:
                return {
                    "name": record["e.name"],
                    "type": record["e.type"] or "Unknown",
                    "description": record["e.description"] or "No description",
                    "attributes": json.loads(record["e.attributes"] or "{}"),
                    "relationships": record["relationships"]
                }
            return {}

    def adaptive_filter_information(self, query: str, hierarchical_info: Dict[int, List[Dict]]) -> List[Dict]:
        """Apply adaptive filtering to retrieved information with layer weighting"""
        analysis_reports = []
        max_layer = max(hierarchical_info.keys(), default=0)

        for layer, layer_info in sorted(hierarchical_info.items(), reverse=True):
            if not layer_info:
                continue

            layer_text = self._format_layer_info(layer_info)
            layer_weight = 1.0 + (layer / max_layer) * 0.5 if max_layer > 0 else 1.0

            filter_prompt = ChatPromptTemplate.from_messages([
                ("system",
                "Analyze the relevance of the provided information for the query. "
                "Respond with ONLY a valid JSON object in this exact format:\n"
                '{"relevance_score": 7.5, "analysis": "brief analysis text", "relevant_content": "most relevant parts"}\n'
                "relevance_score must be a number from 0-10. Do not include any text before or after the JSON."),
                ("human", "Query: {query}\n\nRetrieved Information:\n{info}")
            ])

            filter_chain = LLMChain(llm=self.llm, prompt=filter_prompt)

            response_text = ""  # Initialize to avoid UnboundLocalError
            try:
                response = filter_chain.invoke({"query": query, "info": layer_text})
                response_text = response["text"].strip()
                logging.debug(f"LLM response for layer {layer}: {response_text}")

                # More robust JSON extraction
                json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
                matches = re.findall(json_pattern, response_text, re.DOTALL)

                report = None
                for match in matches:
                    try:
                        potential_report = json.loads(match)
                        # Check if it has the required structure
                        if isinstance(potential_report, dict) and "relevance_score" in potential_report:
                            report = potential_report
                            break
                    except json.JSONDecodeError:
                        continue

                if not report:
                    # Fallback: try to extract values manually
                    relevance_match = re.search(r'"relevance_score":\s*(\d+\.?\d*)', response_text)
                    relevance_score = float(relevance_match.group(1)) if relevance_match else 5.0

                    report = {
                        "relevance_score": relevance_score,
                        "analysis": "Auto-generated analysis due to parsing issues",
                        "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text
                    }

                # Validate and fix the report
                report["relevance_score"] = float(report.get("relevance_score", 5.0))
                report["analysis"] = str(report.get("analysis", "No analysis available"))
                report["relevant_content"] = str(report.get("relevant_content", layer_text))

                # Apply layer weighting
                report["relevance_score"] = min(report["relevance_score"] * layer_weight, 10)
                report["layer"] = layer
                report["original_info"] = layer_info
                analysis_reports.append(report)

            except Exception as e:
                # Robust fallback for any error
                analysis_reports.append({
                    "relevance_score": 3.0 * layer_weight,
                    "analysis": f"Processing fallback due to error: {str(e)[:100]}",
                    "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text,
                    "layer": layer,
                    "original_info": layer_info
                })

        analysis_reports = [r for r in analysis_reports if r.get("relevance_score", 0) >= self.relevance_threshold * 10]
        analysis_reports.sort(key=lambda x: x.get("relevance_score", 0), reverse=True)
        return analysis_reports

    def _format_layer_info(self, layer_info: List[Dict]) -> str:
        """Format layer information for LLM processing"""
        formatted_parts = []
        for item in layer_info:
            if item["type"] == "entity":
                content = item["content"]
                formatted_parts.append(
                    f"Entity: {content.get('name', 'Unknown')}\n"
                    f"Type: {content.get('type', 'Unknown')}\n"
                    f"Description: {content.get('description', 'No description')}\n"
                    f"Attributes: {json.dumps(content.get('attributes', {}))}\n"
                    f"Relationships: {json.dumps(content.get('relationships', []))}\n"
                )
            elif item["type"] == "community":
                content = item["content"]
                formatted_parts.append(
                    f"Community {content.get('id', 'Unknown')} (Layer {content.get('layer', 'Unknown')}):\n"
                    f"Summary: {content.get('summary', 'No summary')}\n"
                    f"Size: {content.get('size', 0)} members\n"
                    f"Members: {', '.join(content.get('members', []))}\n"
                )
        return "\n---\n".join(formatted_parts)

    def generate_response(self, query: str, filtered_info: List[Dict]) -> str:
        """Generate final response using filtered information"""
        if not filtered_info:
            return "I couldn't find relevant information to answer your query."

        context_parts = [report.get("relevant_content", "") for report in filtered_info]
        context = "\n\n".join(context_parts)

        response_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Answer the query based on the provided context. "
             "Cite specific entities or relationships when relevant. "
             "If the context is insufficient, acknowledge the limitations and provide a general answer."),
            ("human", "Question: {query}\n\nContext:\n{context}")
        ])

        response_chain = LLMChain(llm=self.llm, prompt=response_prompt)

        try:
            result = response_chain.invoke({"query": query, "context": context})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating response: {e}")
            return f"Error generating response: {e}"

    def query(self, query: str, k: int = 3) -> Dict[str, Any]:
        """Main query method for ArchRAG pipeline"""
        logging.info(f"Processing query: {query}")

        hierarchical_info = self.retrieve_hierarchical_info(query, k)
        filtered_info = self.adaptive_filter_information(query, hierarchical_info)
        response = self.generate_response(query, filtered_info)

        return {
            "query": query,
            "response": response,
            "hierarchical_info": hierarchical_info,
            "filtered_info": filtered_info,
            "layers_searched": list(hierarchical_info.keys())
        }

class ArchRAGPipeline:
    """Complete ArchRAG Pipeline"""
    
    def __init__(self, pdf_dir: str, neo4j_uri: str, neo4j_username: str,
                 neo4j_password: str):
        # Set cache dir to a path with enough space
        os.environ.setdefault("HF_HOME", str(Path(".cache/huggingface")))
        os.environ.setdefault("TRANSFORMERS_CACHE", str(Path(".cache/huggingface")))
        
        self.pdf_processor = PDFProcessor(pdf_dir)
        self.graph_store = ArchRAGStore(neo4j_uri, neo4j_username, neo4j_password)
        
        # Initialize Llama 3.1 with 4-bit quantization for better performance
        model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        
        try:
            # Configure quantization with more specific settings
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,  # Changed from bfloat16
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            
            # Load tokenizer first
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=HF_TOKEN,
                padding_side="left"
            )
            
            # Set pad token if it doesn't exist
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Load model with minimal device mapping to avoid conflicts
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=HF_TOKEN,
                quantization_config=quantization_config,
                low_cpu_mem_usage=True,
                trust_remote_code=True,
                # Remove device_map and torch_dtype to avoid conflicts with quantization
            )
            
            # Create pipeline with explicit device handling
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False,  # Only return generated text
                device=model.device if hasattr(model, 'device') else 0  # Use model's device
            )
            
            print(f"Model loaded successfully on device: {model.device}")
            
        except Exception as e:
            print(f"Error loading quantized model: {e}")
            print("Falling back to non-quantized model...")
            
            # Fallback: Load without quantization
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=HF_TOKEN,
                padding_side="left"
            )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=HF_TOKEN,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True
            )
            
            pipe = hf_pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )
        
        hf_llm = HuggingFacePipeline(pipeline=pipe)
        self.llm = ChatHuggingFace(llm=hf_llm)  # Wraps for chat template compatibility
        self.triplet_extractor = TripletExtractor(self.llm)
        self.query_engine = ArchRAGQueryEngine(self.graph_store, self.llm)

    def build_knowledge_graph(self, max_docs: Optional[int] = None):
        """Build the complete knowledge graph from PDFs"""
        logging.info("Starting ArchRAG knowledge graph construction...")

        texts = self.pdf_processor.extract_texts()
        if max_docs:
            texts = texts[:max_docs]
        logging.info(f"Extracted text from {len(texts)} documents")

        chunks = self.pdf_processor.create_chunks(texts)
        logging.info(f"Created {len(chunks)} chunks")

        for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
            doc_id = f"doc_{i // 10}_chunk_{i % 10}"
            entities, relationships = self.triplet_extractor.extract_triplets(chunk)
            if entities or relationships:
                self.graph_store.add_triplets(entities, relationships, doc_id)

        self.graph_store.build_communities(self.llm)
        logging.info("ArchRAG knowledge graph construction completed!")

    def query(self, query: str, k: int = 3) -> Dict[str, Any]:
        """Query the ArchRAG system"""
        return self.query_engine.query(query, k)

    def close(self):
        """Clean up resources"""
        self.graph_store.close()

# Utility functions
def analyze_graph_structure(graph_store: ArchRAGStore):
    """Analyze the structure of the built knowledge graph"""
    with graph_store.driver.session() as session:
        entity_count = session.run("MATCH (e:Entity) RETURN count(e) as count").single()["count"]
        rel_count = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) as count").single()["count"]
        entity_types = session.run(
            "MATCH (e:Entity) RETURN e.type, count(e) as count ORDER BY count DESC"
        )

        print(f"\nGraph Structure Analysis:")
        print(f"Total Entities: {entity_count}")
        print(f"Total Relationships: {rel_count}")
        print(f"Entity Types:")
        for record in entity_types:
            print(f"  {record['e.type']}: {record['count']}")

        print(f"\nHierarchical Communities:")
        for layer, communities in graph_store.hierarchical_communities.items():
            print(f"Layer {layer}: {len(communities)} communities")
            for comm in communities:
                print(f"  {comm['id']}: {comm['size']} members")

def export_graph_data(graph_store: ArchRAGStore, output_file: str = "archrag_export.json"):
    """Export graph data to JSON file"""
    export_data = {
        "entities": [],
        "relationships": [],
        "hierarchical_communities": graph_store.hierarchical_communities
    }

    with graph_store.driver.session() as session:
        entities = session.run("MATCH (e:Entity) RETURN e")
        for record in entities:
            entity = dict(record["e"])
            export_data["entities"].append(entity)

        relationships = session.run("MATCH ()-[r:RELATION]->() RETURN r")
        for record in relationships:
            rel = dict(record["r"])
            export_data["relationships"].append(rel)

    with open(output_file, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)

    print(f"Graph data exported to {output_file}")

def main():
    """Main execution function"""
    config = {
        #"pdf_dir": str(PDF_DIR),
        "pdf_dir": str(PDF_DIR),
        "neo4j_uri": NEO4J_URI,
        "neo4j_username": NEO4J_USERNAME,
        "neo4j_password": NEO4J_PASSWORD,
    }

    pipeline = ArchRAGPipeline(**config)

    try:
        pipeline.build_knowledge_graph(max_docs=3)
        queries = [
            "What are the main concepts discussed in the documents?",
            "How are different entities related to each other?"
        ]

        print("\n" + "="*50)
        print("QUERYING ARCHRAG SYSTEM")
        print("="*50)

        for query in queries:
            print(f"\nQuery: {query}")
            result = pipeline.query(query)
            print(f"Response: {result['response']}")
            print(f"Layers searched: {result['layers_searched']}")
            print("-" * 30)

        analyze_graph_structure(pipeline.graph_store)
        export_graph_data(pipeline.graph_store)

    except Exception as e:
        logging.error(f"Error in main execution: {e}", exc_info=True)
        raise
    finally:
        pipeline.close()

if __name__ == "__main__":
    main()

In [ ]:
!pip uninstall langchain-core langchain-huggingface -y
!pip install langchain-core langchain-huggingface

In [ ]:
%pip install --upgrade langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters

In [ ]:
!pip install langchain-classic

In [ ]:
# First remove any old versions
!pip uninstall -y llama_index gpt_index

# Install the latest full package (with integrations like Ollama, HuggingFace, etc.)
!pip install -U llama-index
!pip install llama-index-llms-huggingface llama-index-llms-ollama
!pip install rouge-score


In [ ]:
# Install LangChain and the community LLM integrations
!pip install -U langchain langchain_community


In [ ]:
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator
)

# These imports will now work
from llama_index.llms.ollama import Ollama
from llama_index.llms.huggingface import HuggingFaceLLM
from rouge_score import rouge_scorer



In [ ]:
# Install specific LlamaIndex LLM packages
!pip install llama-index-llms-ollama llama-index-llms-huggingface

# Or install all LlamaIndex LLM integrations
!pip install 'llama-index[llms]'

In [ ]:
!pip install -U llama-index


In [ ]:
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator
)

# Community LLM connectors
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.ollama import Ollama


In [ ]:
!pip show transformers

In [ ]:
!pip install pandas

In [ ]:
!pip install mistral-common[sentencepiece]
!pip install mistral-common

In [ ]:
import os
import json
import pandas as pd
import logging
import matplotlib.pyplot as plt
from tqdm import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from typing import List, Dict, Any
import nltk

from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.output_parsers import StrOutputParser
from neo4j import GraphDatabase
from transformers import AutoTokenizer, AutoModelForCausalLM,  BitsAndBytesConfig
import torch
from transformers import pipeline as hf_pipeline
# Ensure NLTK punkt is downloaded
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception as e:
    logging.error(f"Failed to download NLTK punkt: {e}")
    raise

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set Hugging Face cache directory
os.environ.setdefault("HF_HOME", str(Path(".cache/huggingface")))
os.environ.setdefault("TRANSFORMERS_CACHE", str(Path(".cache/huggingface")))

def check_neo4j_graph(neo4j_uri: str, neo4j_username: str, neo4j_password: str) -> Dict[str, Any]:
    """Check the Neo4j database for nodes, relationships, and diabetes-related entities."""
    try:
        driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_username, neo4j_password))
        with driver.session() as session:
            entity_result = session.run("MATCH (e:Entity) RETURN count(e) AS entity_count")
            entity_count = entity_result.single()["entity_count"]
            rel_result = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) AS rel_count")
            rel_count = rel_result.single()["rel_count"]
            sample_entities = session.run(
                "MATCH (e:Entity) RETURN e.name, e.type, e.description LIMIT 5"
            ).data()
            sample_rels = session.run(
                "MATCH (e1)-[r:RELATION]->(e2) RETURN e1.name, r.type, e2.name LIMIT 5"
            ).data()
            diabetes_entities = session.run(
                "MATCH (e:Entity) WHERE toLower(e.name) CONTAINS 'diabetes' OR toLower(e.description) CONTAINS 'diabetes' "
                "RETURN e.name, e.type, e.description LIMIT 10"
            ).data()
            family_history_entities = session.run(
                "MATCH (e:Entity) WHERE toLower(e.name) CONTAINS 'family history' OR toLower(e.description) CONTAINS 'family history' "
                "RETURN e.name, e.type, e.description LIMIT 10"
            ).data()
        driver.close()
        logging.info(f"Neo4j Graph Stats: {entity_count} entities, {rel_count} relationships")
        logging.info(f"Sample Entities: {sample_entities}")
        logging.info(f"Sample Relationships: {sample_rels}")
        logging.info(f"Diabetes-Related Entities: {diabetes_entities}")
        logging.info(f"Family History Entities: {family_history_entities}")
        return {
            "entity_count": entity_count,
            "rel_count": rel_count,
            "sample_entities": sample_entities,
            "sample_relationships": sample_rels,
            "diabetes_entities": diabetes_entities,
            "family_history_entities": family_history_entities
        }
    except Exception as e:
        logging.error(f"Neo4j connection error: {e}")
        return {
            "entity_count": 0,
            "rel_count": 0,
            "sample_entities": [],
            "sample_relationships": [],
            "diabetes_entities": [],
            "family_history_entities": []
        }

def sample_dataset(dataset_path: str, num_samples: int = 3) -> List[Dict]:
    """Sample entries from the evaluation dataset."""
    try:
        with open(dataset_path, 'r') as f:
            data = [json.loads(line) for line in f]
        samples = data[:min(num_samples, len(data))]
        for i, sample in enumerate(samples):
            logging.info(f"Dataset Sample {i+1}: {sample}")
        return samples
    except Exception as e:
        logging.error(f"Error sampling dataset: {e}")
        return []

def test_query_engine(query_engine, dataset_path: str) -> Dict[str, Any]:
    """Test the query engine with questions from the dataset."""
    samples = sample_dataset(dataset_path, num_samples=2)
    test_queries = [sample["question"] for sample in samples]
    results = {}
    for test_query in test_queries:
        try:
            result = query_engine.query(test_query)
            logging.info(f"Test Query: {test_query}")
            logging.info(f"Query Engine Output Type: {type(result)}")
            logging.info(f"Query Engine Output: {result}")
            if isinstance(result, dict):
                response_text = result.get('response', 'No response')
                hierarchical_info = result.get('hierarchical_info', {})
                logging.info(f"Response: {response_text}")
                logging.info(f"Hierarchical Info: {hierarchical_info}")
            else:
                response_text = str(result)
                logging.info(f"Response: {response_text}")
                logging.info("Hierarchical Info: Not available (string response)")
            results[test_query] = result
        except Exception as e:
            logging.error(f"Error testing query '{test_query}': {e}")
            results[test_query] = {}
    return results

def load_eval_dataset(json_file_path: str, max_queries: int = 5) -> List[Dict]:
    """Load evaluation dataset from JSONL file, limited to max_queries."""
    try:
        with open(json_file_path, 'r') as f:
            data = [json.loads(line) for line in f]
        data = data[:min(max_queries, len(data))]
        logging.info(f"Loaded {len(data)} evaluation samples (limited to {max_queries})")
        return data
    except Exception as e:
        logging.error(f"Error loading dataset: {e}")
        raise

def init_evaluators(pipeline, model_name: str = "mistralai/Mixtral-7B-Instruct-v0.1") -> tuple:
    """Initialize evaluators, reusing pipeline's LLM if available, else load a smaller model."""
    try:
        # Try to reuse pipeline's LLM
        if hasattr(pipeline, 'llm') and pipeline.llm is not None:
            logging.info("Reusing LLM from pipeline for evaluation...")
            llm = pipeline.llm
        else:
            logging.info(f"Pipeline LLM not found, initializing {model_name} with 4-bit quantization...")
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                token=HF_TOKEN,
                quantization_config=quantization_config,
                device_map="auto",
                low_cpu_mem_usage=True
            )
            



            text_generator = hf_pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=128,
                temperature=0.01,
                do_sample=False,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )
            llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_generator))

        class CustomFaithfulnessEvaluator:
            def __init__(self, llm):
                self.llm = llm
                self.prompt = ChatPromptTemplate.from_messages([
                    ("system", "Determine if the response is faithful to the provided contexts. "
                               "Return a JSON object with 'passing' (boolean) and 'feedback' (string)."),
                    ("human", "Query: {query}\nResponse: {response}\nContexts: {contexts}")
                ])
                self.chain = self.prompt | self.llm | StrOutputParser()

            def evaluate(self, query, response, contexts):
                try:
                    result = json.loads(self.chain.invoke({
                        "query": query,
                        "response": response,
                        "contexts": "\n".join(contexts)
                    }))
                    return type('Result', (), {
                        'passing': result.get('passing', False),
                        'feedback': result.get('feedback', 'No feedback')
                    })()
                except Exception as e:
                    logging.warning(f"Faithfulness evaluation failed: {e}")
                    return type('Result', (), {'passing': False, 'feedback': f"Error: {e}"})()

        class CustomRelevancyEvaluator:
            def __init__(self, llm):
                self.llm = llm
                self.prompt = ChatPromptTemplate.from_messages([
                    ("system", "Determine if the response is relevant to the query based on the contexts. "
                               "Return a JSON object with 'passing' (boolean) and 'feedback' (string)."),
                    ("human", "Query: {query}\nResponse: {response}\nContexts: {contexts}")
                ])
                self.chain = self.prompt | self.llm | StrOutputParser()

            def evaluate(self, query, response, contexts):
                try:
                    result = json.loads(self.chain.invoke({
                        "query": query,
                        "response": response,
                        "contexts": "\n".join(contexts)
                    }))
                    return type('Result', (), {
                        'passing': result.get('passing', False),
                        'feedback': result.get('feedback', 'No feedback')
                    })()
                except Exception as e:
                    logging.warning(f"Relevancy evaluation failed: {e}")
                    return type('Result', (), {'passing': False, 'feedback': f"Error: {e}"})()

        class CustomCorrectnessEvaluator:
            def __init__(self, llm):
                self.llm = llm
                self.prompt = ChatPromptTemplate.from_messages([
                    ("system", "Evaluate the correctness of the response compared to the reference answer. "
                               "Return a JSON object with 'score' (0-5), 'passing' (boolean, true if score >= 4), "
                               "and 'feedback' (string)."),
                    ("human", "Query: {query}\nResponse: {response}\nReference: {reference}")
                ])
                self.chain = self.prompt | self.llm | StrOutputParser()

            def evaluate(self, query, response, reference):
                try:
                    result = json.loads(self.chain.invoke({
                        "query": query,
                        "response": response,
                        "reference": reference
                    }))
                    score = result.get('score', 0.0)
                    return type('Result', (), {
                        'score': score,
                        'passing': score >= 4,
                        'feedback': result.get('feedback', 'No feedback')
                    })()
                except Exception as e:
                    logging.warning(f"Correctness evaluation failed: {e}")
                    return type('Result', (), {
                        'score': 0.0,
                        'passing': False,
                        'feedback': f"Error: {e}"
                    })()

        faithfulness_evaluator = CustomFaithfulnessEvaluator(llm)
        relevancy_evaluator = CustomRelevancyEvaluator(llm)
        correctness_evaluator = CustomCorrectnessEvaluator(llm)
        logging.info("Successfully initialized evaluators")
        return faithfulness_evaluator, relevancy_evaluator, correctness_evaluator, llm
    except Exception as e:
        logging.error(f"Error initializing evaluators: {e}")
        raise

def compute_rouge_bleu(response: str, reference: str) -> Dict[str, float]:
    """Compute ROUGE and BLEU scores."""
    try:
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_scores = scorer.score(reference, response)
        rouge_l = rouge_scores['rougeL'].fmeasure
        reference_tokens = [nltk.word_tokenize(reference.lower())]
        response_tokens = nltk.word_tokenize(response.lower())
        bleu_score = sentence_bleu(reference_tokens, response_tokens, weights=(0.25, 0.25, 0.25, 0.25),
                                  smoothing_function=SmoothingFunction().method1)
        return {"rouge_l": rouge_l, "bleu_4": bleu_score}
    except Exception as e:
        logging.warning(f"Error computing ROUGE/BLEU: {e}")
        return {"rouge_l": 0.0, "bleu_4": 0.0}

def evaluate_node_relevance(query: str, hierarchical_info: Dict[int, List[Dict]], llm) -> float:
    """Evaluate relevance of retrieved graph nodes to the query."""
    try:
        relevant_nodes = 0
        total_nodes = 0
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Determine if the provided node content is relevant to the query. "
                       "Return 'yes' or 'no'."),
            ("human", "Query: {query}\nNode Content: {content}")
        ])
        chain = prompt | llm | StrOutputParser()
        for layer, nodes in hierarchical_info.items():
            for node in nodes:
                total_nodes += 1
                content = node.get('content', {}).get('summary', node.get('content', {}).get('description', ''))
                if not content:
                    continue
                result = chain.invoke({"query": query, "content": content}).strip().lower()
                if result == "yes":
                    relevant_nodes += 1
        return relevant_nodes / total_nodes if total_nodes > 0 else 0.0
    except Exception as e:
        logging.warning(f"Error evaluating node relevance: {e}")
        return 0.0

def evaluate_query(query_engine, query_data: Dict, evaluators: tuple, llm) -> Dict[str, Any]:
    """Evaluate a single query against the ArchRAG query engine."""
    query = query_data["question"]
    reference = query_data["answer"]
    context = query_data.get("context", "")
    try:
        result = query_engine.query(query)
        if isinstance(result, dict):
            response_text = result.get("response", "")
            hierarchical_info = result.get("hierarchical_info", {})
        else:
            response_text = str(result)
            hierarchical_info = {}
        if not response_text or response_text.startswith("I couldn't find relevant information"):
            logging.warning(f"Empty or invalid response for query: {query}")
            response_text = ""
            logging.info(f"Query Result: {result}")
            logging.info(f"Retrieved Nodes: {hierarchical_info}")
        faithfulness_evaluator, relevancy_evaluator, correctness_evaluator = evaluators
        eval_contexts = []
        if hierarchical_info:
            for layer, nodes in hierarchical_info.items():
                for node in nodes:
                    content = node.get('content', {})
                    content = content.get('summary', content.get('description', '')) if isinstance(content, dict) else str(content)
                    if content:
                        eval_contexts.append(content)
        if not eval_contexts:
            eval_contexts = [context] if context else ["No context available"]
        faithfulness_result = faithfulness_evaluator.evaluate(
            query=query,
            response=response_text,
            contexts=eval_contexts
        )
        relevancy_result = relevancy_evaluator.evaluate(
            query=query,
            response=response_text,
            contexts=eval_contexts
        )
        source_relevancy_results = []
        if hierarchical_info:
            for layer, nodes in hierarchical_info.items():
                for node in nodes:
                    content = node.get('content', {})
                    content = content.get('summary', content.get('description', '')) if isinstance(content, dict) else str(content)
                    if not content:
                        continue
                    try:
                        node_result = relevancy_evaluator.evaluate(
                            query=query,
                            response=response_text,
                            contexts=[content]
                        )
                        source_relevancy_results.append(node_result.passing)
                    except Exception as e:
                        logging.warning(f"Error evaluating node relevancy: {e}")
                        continue
        source_relevancy_score = (sum(source_relevancy_results) / len(source_relevancy_results)
                                 if source_relevancy_results else 0.0)
        correctness_result = correctness_evaluator.evaluate(
            query=query,
            response=response_text,
            reference=reference
        )
        rouge_bleu_scores = compute_rouge_bleu(response_text, reference)
        node_precision = evaluate_node_relevance(query, hierarchical_info, llm) if hierarchical_info else 0.0
        community_coverage = 0.0
        if hierarchical_info:
            valid_layers = [l for l, nodes in hierarchical_info.items() if nodes]
            community_coverage = len(valid_layers) / (len(hierarchical_info) or 1)
        return {
            "query": query,
            "response": response_text,
            "reference": reference,
            "faithfulness_score": 1 if faithfulness_result.passing else 0,
            "faithfulness_feedback": faithfulness_result.feedback,
            "relevancy_score": 1 if relevancy_result.passing else 0,
            "relevancy_feedback": relevancy_result.feedback,
            "source_relevancy_score": source_relevancy_score,
            "correctness_score": correctness_result.score,
            "correctness_passing": 1 if correctness_result.passing else 0,
            "correctness_feedback": correctness_result.feedback,
            "rouge_l": rouge_bleu_scores["rouge_l"],
            "bleu_4": rouge_bleu_scores["bleu_4"],
            "node_precision": node_precision,
            "community_coverage": community_coverage
        }
    except Exception as e:
        logging.error(f"Error evaluating query '{query}': {e}")
        import traceback
        logging.error(f"Traceback: {traceback.format_exc()}")
        return {
            "query": query,
            "response": "",
            "reference": reference,
            "faithfulness_score": 0,
            "faithfulness_feedback": f"Evaluation error: {e}",
            "relevancy_score": 0,
            "relevancy_feedback": f"Evaluation error: {e}",
            "source_relevancy_score": 0.0,
            "correctness_score": 0.0,
            "correctness_passing": 0,
            "correctness_feedback": f"Evaluation error: {e}",
            "rouge_l": 0.0,
            "bleu_4": 0.0,
            "node_precision": 0.0,
            "community_coverage": 0.0
        }

def create_visualizations(results_df: pd.DataFrame, metrics: Dict[str, float]) -> None:
    """Create visualizations for evaluation metrics."""
    try:
        valid_metrics = {k: v for k, v in metrics.items() if not pd.isna(v)}
        metric_names = list(valid_metrics.keys())
        metric_values = list(valid_metrics.values())
        plt.figure(figsize=(12, 6))
        bars = plt.bar(metric_names, metric_values)
        plt.ylim(0, 1.1)
        plt.title('ArchRAG Evaluation Metrics')
        plt.ylabel('Score')
        plt.xticks(rotation=45)
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                     f'{height:.2f}', ha='center', va='bottom')
        plt.tight_layout()
        plt.savefig('./workspace/test/archrag_evaluation_metrics_test.png')
        plt.close()
        if 'correctness_score' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(results_df['correctness_score'].dropna(), bins=10, alpha=0.7)
            plt.title('Distribution of Correctness Scores')
            plt.xlabel('Correctness Score')
            plt.ylabel('Count')
            plt.savefig('./workspace/test/correctness_distribution_test.png')
            plt.close()
        if 'node_precision' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(results_df['node_precision'].dropna(), bins=10, alpha=0.7)
            plt.title('Distribution of Node Retrieval Precision')
            plt.xlabel('Node Precision')
            plt.ylabel('Count')
            plt.savefig('./workspace/test/node_precision_distribution_test.png')
            plt.close()
        if 'rouge_l' in results_df.columns and 'node_precision' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.scatter(results_df['rouge_l'], results_df['node_precision'], alpha=0.5)
            plt.title('ROUGE-L vs Node Retrieval Precision')
            plt.xlabel('ROUGE-L Score')
            plt.ylabel('Node Precision')
            plt.savefig('./workspace/test/rouge_vs_node_test.png')
            plt.close()
    except Exception as e:
        logging.error(f"Error creating visualizations: {e}")

def evaluate_archrag_system(pipeline, dataset_path: str, config: Dict[str, str]) -> tuple:
    """Evaluate the ArchRAG system using the existing knowledge graph and dataset."""
    logging.info("Starting ArchRAG evaluation...")
    pdf_texts = []  # Skip PDF reading
    dataset_samples = sample_dataset(dataset_path)
    graph_stats = check_neo4j_graph(config["neo4j_uri"], config["neo4j_username"], config["neo4j_password"])
    query_engine = pipeline.query_engine
    test_query_engine(query_engine, dataset_path)
    eval_dataset = load_eval_dataset(dataset_path, max_queries=5)
    evaluators = init_evaluators(pipeline)
    llm = evaluators[3]
    results = []
    for query_data in tqdm(eval_dataset, desc="Evaluating queries"):
        result = evaluate_query(query_engine, query_data, evaluators[:3], llm)
        results.append(result)
    results_df = pd.DataFrame(results)
    metrics = {
        "Faithfulness": results_df["faithfulness_score"].mean(),
        "Relevancy": results_df["relevancy_score"].mean(),
        "Source Relevancy": results_df["source_relevancy_score"].mean() if not results_df["source_relevancy_score"].empty else 0.0,
        "Correctness Score": results_df["correctness_score"].mean(),
        "Correctness Pass Rate": results_df["correctness_passing"].mean(),
        "ROUGE-L": results_df["rouge_l"].mean(),
        "BLEU-4": results_df["bleu_4"].mean(),
        "Node Precision": results_df["node_precision"].mean(),
        "Community Coverage": results_df["community_coverage"].mean()
    }
    metrics_df = pd.DataFrame({
        "Metric": list(metrics.keys()),
        "Score": list(metrics.values())
    })
    create_visualizations(results_df, metrics)
    results_df.to_csv("./workspace/test/archrag_evaluation_detailed_results_test.csv", index=False)
    metrics_df.to_csv("./workspace/test/archrag_evaluation_metrics_test.csv", index=False)
    logging.info("\nArchRAG Evaluation Summary:")
    logging.info(metrics_df.to_string(index=False))
    logging.info("\nDetailed results saved to './workspace/test/hagrag_evaluation_detailed_results_test.csv'")
    logging.info("Metrics summary saved to './workspace/test/hagrag_evaluation_metrics_test.csv'")
    logging.info("Visualizations saved in './workspace/test/'")
    return results_df, metrics_df, graph_stats, pdf_texts

if __name__ == "__main__":
    config = {
        "pdf_dir": str(PDF_DIR),
        "neo4j_uri": NEO4J_URI,
        "neo4j_username": NEO4J_USERNAME,
        "neo4j_password": NEO4J_PASSWORD,
    }
    dataset_path = str(DATASET_PATH)
    pipeline = ArchRAGPipeline(**config)
    # Ensure 'pipeline' is your initialized ArchRAGPipeline object
    try:
        results_df, metrics_df, graph_stats, pdf_texts = evaluate_archrag_system(
            pipeline=pipeline,
            dataset_path=dataset_path,
            config=config)
        
    


    except Exception as e:
        logging.error(f"Evaluation failed: {e}")
        import traceback
        logging.error(f"Traceback: {traceback.format_exc()}")

In [ ]:
# ============================
# Reason+Explain (HF Transformers, no OpenAI)
# - Robust HF loader (4-bit → 8-bit → fp16 → CPU)
# - Safe Neo4j: probe driver; fall back to NX if connection is flaky
# - RGP guardrails: validate edge triples to avoid unpack errors
# - Reuses existing KG/communities (won’t rebuild)
# ============================

import os, json, math, re, logging
from typing import Any, Dict, List, Tuple
import numpy as np
import networkx as nx
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline as hf_pipeline,
)
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# ---------- Robust HF chat LLM loader with fallbacks ----------
_HF_CHAT_CACHE: dict[str, ChatHuggingFace] = {}

def _mk_chat_from(model, tokenizer, max_new_tokens, temperature, top_p, rep_penalty):
    gen = hf_pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=top_p,
        repetition_penalty=rep_penalty,
        return_full_text=False,
    )
    return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen))

def build_hf_chat_llm(
    model_id: str = "mistralai/Mistral-7B-Instruct-v0.3",
    *,
    prefer_4bit: bool = True,
    max_new_tokens: int = 256,
    temperature: float = 0.2,
    top_p: float = 0.9,
    rep_penalty: float = 1.1,
):
    if model_id in _HF_CHAT_CACHE:
        return _HF_CHAT_CACHE[model_id]

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    def try_load(msg, **kwargs):
        logging.info(msg)
        m = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            **kwargs,
        )
        return _mk_chat_from(m, tok, max_new_tokens, temperature, top_p, rep_penalty)

    if prefer_4bit:
        # 4-bit (no device_map)
        try:
            chat = try_load(
                f"[HF] Trying 4-bit (no device_map) for {model_id}",
                quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                ),
            )
            _HF_CHAT_CACHE[model_id] = chat
            return chat
        except Exception as e:
            logging.warning(f"[HF] 4-bit (no device_map) failed: {e}")
        # 4-bit (device_map='auto')
        try:
            chat = try_load(
                f"[HF] Trying 4-bit (device_map='auto') for {model_id}",
                quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                ),
                device_map="auto",
            )
            _HF_CHAT_CACHE[model_id] = chat
            return chat
        except Exception as e:
            logging.warning(f"[HF] 4-bit (device_map='auto') failed: {e}")
        # 8-bit
        try:
            chat = try_load(
                f"[HF] Trying 8-bit (device_map='auto') for {model_id}",
                quantization_config=BitsAndBytesConfig(load_in_8bit=True),
                device_map="auto",
            )
            _HF_CHAT_CACHE[model_id] = chat
            return chat
        except Exception as e:
            logging.warning(f"[HF] 8-bit failed: {e}")

    # fp16
    try:
        chat = try_load(
            f"[HF] Trying fp16 (device_map='auto') for {model_id}",
            torch_dtype=torch.float16,
            device_map="auto",
        )
        _HF_CHAT_CACHE[model_id] = chat
        return chat
    except Exception as e:
        logging.warning(f"[HF] fp16 failed: {e}")

    # CPU
    chat = try_load(
        f"[HF] Falling back to CPU for {model_id}",
        torch_dtype=torch.float32,
    )
    _HF_CHAT_CACHE[model_id] = chat
    return chat

# ---------- Robust chat adapter ----------
class _RobustLLMAdapter:
    def __init__(self, llm): self.llm=llm; self._has_invoke=hasattr(llm,"invoke") and callable(llm.invoke)
    def _as_text(self, obj):
        if obj is None: return ""
        try:
            from langchain_core.messages import AIMessage
            if isinstance(obj, AIMessage): return getattr(obj,"content","") or ""
        except Exception: pass
        if isinstance(obj, dict):
            for k in ("content","text","output_str"):
                if k in obj and isinstance(obj[k],str): return obj[k]
        if isinstance(obj, list) and obj and isinstance(obj[0], dict):
            return obj[0].get("generated_text") or obj[0].get("text") or json.dumps(obj[0])
        if isinstance(obj, str): return obj
        return getattr(obj,"content",None) or str(obj)
    def call(self, prompt: str, context: str|None=None) -> str:
        try:
            if self._has_invoke:
                from langchain_core.messages import SystemMessage, HumanMessage
                msgs=[SystemMessage(content="You are precise and follow formatting."),
                      HumanMessage(content=prompt if not context else f"{prompt}\n\nCONTEXT:\n{context}")]
                return self._as_text(self.llm.invoke(msgs))
            for m in ("predict","generate","call","invoke","__call__"):
                if hasattr(self.llm,m) and callable(getattr(self.llm,m)):
                    return self._as_text(getattr(self.llm,m)(prompt))
            return ""
        except Exception as e:
            logging.warning(f"LLM call failed: {e}"); return ""

# ---------- Store API shims ----------
def _ensure_store_api(store):
    if not hasattr(store,"embed"):
        def _embed(t:str)->np.ndarray:
            v=np.asarray(store.embedding_model.encode(t),dtype=np.float32)
            v/=(np.linalg.norm(v)+1e-12); return v
        setattr(store,"embed",_embed)
    if not hasattr(store,"search"):
        def _search(vec:np.ndarray,k:int=12)->List[tuple]:
            hits=[]
            for nid, emb in getattr(store,"entity_embeddings",{}).items():
                if emb is None: continue
                e=np.asarray(emb,dtype=np.float32); e/=(np.linalg.norm(e)+1e-12)
                hits.append((nid,float(np.dot(vec,e))))
            hits.sort(key=lambda x:-x[1]); return hits[:k]
        setattr(store,"search",_search)
    if not hasattr(store,"get_layer"):
        setattr(store,"get_layer", lambda node_id: int(node_id.split("_")[0][1:]) if isinstance(node_id,str) and node_id.startswith("L") and "_C" in node_id else 0)
    if not hasattr(store,"get_text"):
        def _get_text(node_id:str)->str:
            try:
                for _,comms in (store.hierarchical_communities or {}).items():
                    for c in comms:
                        if c.get("id")==node_id: return c.get("summary") or node_id
            except Exception: pass
            try:
                with store.driver.session() as s:
                    rec=s.run("""MATCH (e:Entity {name:$n}) RETURN e.name as name, e.type as type, e.description as d, e.attributes as a""", n=node_id).single()
                    if rec:
                        attrs=rec["a"]; attrs_s=attrs if isinstance(attrs,str) else json.dumps(attrs or {})
                        return f"{rec['name']} {rec.get('type') or ''} {rec.get('d') or ''} {attrs_s}"
            except Exception: pass
            return node_id
        setattr(store,"get_text",_get_text)
    if not hasattr(store,"neighbors"):
        def _neighbors(node_id:str)->List[str]:
            try:
                for _,comms in (store.hierarchical_communities or {}).items():
                    for c in comms:
                        if c.get("id")==node_id:
                            mems=c.get("members",[]) or []; down=getattr(store.chnsw_index,"inter_layer_links",{}).get(node_id)
                            return (mems+([down] if down else []))[:10]
            except Exception: pass
            try:
                G=store._create_nx_graph()
                return list(G.neighbors(node_id)) if node_id in G.nodes else []
            except Exception: return []
        setattr(store,"neighbors",_neighbors)

# ---------- Neo4j driver probe (avoid defunct connection errors) ----------
def _safe_neo4j_driver(store):
    drv = getattr(store, "driver", None)
    if drv is None: return None
    try:
        with drv.session() as s:
            _ = s.run("RETURN 1 as ok").single()
        return drv
    except Exception as e:
        logging.warning(f"[Neo4j] driver probe failed ({e}); falling back to NX-only verification.")
        return None

# ---------- JSON helpers ----------
_JSON_FENCE = re.compile(r"```(?:json)?\s*(.+?)\s*```", re.DOTALL)
def _extract_first_json(raw: str):
    if not raw: return None
    m=_JSON_FENCE.search(raw)
    if m:
        try: return json.loads(m.group(1).strip())
        except Exception: pass
    for opener,closer in [("[","]"),("{","}")]:
        s=raw.find(opener)
        while s!=-1:
            depth=0
            for i in range(s,len(raw)):
                ch=raw[i]; depth += (ch==opener) - (ch==closer)
                if depth==0:
                    cand=raw[s:i+1]
                    try: return json.loads(cand)
                    except Exception: break
            s=raw.find(opener,s+1)
    return None

def _llm_json_strict(llm_or_adapter, prompt: str, fallback):
    adapter = llm_or_adapter if isinstance(llm_or_adapter,_RobustLLMAdapter) else _RobustLLMAdapter(llm_or_adapter)
    raw = adapter.call(prompt)
    try: return json.loads(raw)
    except Exception:
        js=_extract_first_json(raw)
        if js is None:
            logging.warning("Expected JSON, got: %s", (raw or "")[:300]); return fallback
        return js

# ---------- HRR / CoGT ----------
def hrr_propose_hypotheses(llm, query, n=3)->List[str]:
    js=_llm_json_strict(llm,
        f"Return ONLY a JSON array of N DISTINCT and TESTABLE answer hypotheses (short strings).\nN={n}\nQuery:\n{query}", [])
    hyps=[str(h).strip() for h in js if isinstance(h,(str,int,float))] if isinstance(js,list) else []
    uniq, seen=[], set()
    for h in hyps:
        k=h.lower()
        if h and k not in seen: uniq.append(h); seen.add(k)
    while len(uniq)<n: uniq.append(f"{query.strip().rstrip('?')} — variant {len(uniq)+1}")
    return uniq[:n]

def hrr_pool_per_hypothesis(store, query:str, hyps:List[str], k=12):
    qv=store.embed(query); out={}
    for h in hyps:
        hv=store.embed(h); mix=0.5*qv+0.5*hv
        out[h]=store.search(mix,k=k)
    return out

def layerwise_filter(store, results, threshold=0.05):
    kept=[r for r in results if r[1]>=threshold]; kept.sort(key=lambda x:-x[1])
    keptinfo=[]
    for rank,(nid,score) in enumerate(kept):
        keptinfo.append({"id":nid,"type":"node","layer":store.get_layer(nid),"relevancescore":float(score),"is_decisive":rank<3,"content":{"summary":store.get_text(nid)}})
    layers={}
    for it in keptinfo: layers.setdefault(it["layer"],[]).append(it)
    return dict(sorted(layers.items(), reverse=True))

def cogt_reason(llm, query, filtered_layers):
    blocks=[]
    for layer,items in sorted(filtered_layers.items(), reverse=True):
        lines=[f"[{it['id']}] {it['content'].get('summary','')[:240]}" for it in items[:6]]
        if lines: blocks.append(f"Layer {layer}:\n"+"\n".join(lines))
    joined="\n\n".join(blocks[:4]) or "NO EVIDENCE"
    prompt=("Return ONLY VALID JSON (no prose):\n"
            '[{"node_id":"...", "node_type":"Entity|Community", "claim":"...", '
            '"supporting_snippets":["..."], "confidence":0.0, "expected_counter_evidence":"..."}]\n\n'
            f"QUERY:\n{query}\n\nGRAPH EVIDENCE:\n{joined}")
    js=_llm_json_strict(llm,prompt,[])
    return js if isinstance(js,list) else []

def infer_paths_from_cogt(cogt_steps, filtered_layers=None):
    if not isinstance(cogt_steps,list) or (cogt_steps and not isinstance(cogt_steps[0],dict)): cogt_steps=[]
    ids=[s.get("node_id") for s in cogt_steps if isinstance(s,dict) and s.get("node_id")]
    if ids: return [ids]
    if filtered_layers:
        flat=[]
        for _,items in sorted(filtered_layers.items(), reverse=True): flat.extend(items)
        top=[it["id"] for it in sorted(flat, key=lambda x:-x.get("relevancescore",0.0))[:3]]
        if top: return [top]
    return []

# ---------- RGP with strict validation ----------
def _coerce_str_list(x):
    if not isinstance(x, list): return []
    return [str(v) for v in x if isinstance(v,(str,int,float))]
def _coerce_edge_triples(x):
    """Return only triples (a,rel,b) as strings; ignore malformed edges."""
    if not isinstance(x, list): return []
    triples=[]
    for item in x:
        if isinstance(item,(list,tuple)) and len(item)==3:
            a,rel,b = item
            if all(isinstance(v,(str,int,float)) for v in (a,rel,b)):
                triples.append((str(a),str(rel),str(b)))
    return triples

def rgp_emit_and_verify(llm, query, cogt_steps, nxG=None, neo4j_driver=None):
    prog=_llm_json_strict(llm, f"""Return ONLY JSON as:
{{"require_nodes": [], "require_edges": [], "allow_alternatives": {{}}, "forbid_edges": []}}
Steps:
{json.dumps(cogt_steps)}
Query: {query}
""", {"require_nodes": [], "require_edges": [], "allow_alternatives": {}, "forbid_edges": []})

    # Sanitize shapes to avoid unpack errors
    req_nodes = _coerce_str_list(prog.get("require_nodes", []))
    req_edges = _coerce_edge_triples(prog.get("require_edges", []))
    forbid_edges = _coerce_edge_triples(prog.get("forbid_edges", []))
    allow_alts = prog.get("allow_alternatives", {})
    if not isinstance(allow_alts, dict): allow_alts = {}

    ok_nodes=ok_edges=ok_forbid=True

    # Probe Neo4j connectivity; if bad, force NX fallback
    if neo4j_driver is not None:
        try:
            with neo4j_driver.session() as _s:
                _ = _s.run("RETURN 1 as ok").single()
        except Exception as e:
            logging.warning(f"Neo4j verification failed: {e}; falling back to NX.")
            neo4j_driver=None

    # Neo4j path
    if neo4j_driver is not None:
        try:
            with neo4j_driver.session() as sess:
                for n in req_nodes:
                    c = sess.run("MATCH (x:Entity {name:$n}) RETURN count(x) as c", n=n).single()["c"]
                    if c == 0: ok_nodes=False; break
                if ok_nodes:
                    for (a, rel, b) in req_edges:
                        alts = [rel] + (allow_alts.get(rel, []) if isinstance(allow_alts.get(rel, []), list) else [])
                        cond = " OR ".join([f"(a)-[:{r}]->(b)" for r in alts])
                        q = f"MATCH (a:Entity {{name:$a}}),(b:Entity {{name:$b}}) WHERE {cond} RETURN count(*) as c"
                        c = sess.run(q, a=a, b=b).single()["c"]
                        if c == 0: ok_edges=False; break
                if ok_edges:
                    for (a, rel, b) in forbid_edges:
                        q = f"MATCH (a:Entity {{name:$a}})-[:{rel}]->(b:Entity {{name:$b}}) RETURN count(*) as c"
                        c = sess.run(q, a=a, b=b).single()["c"]
                        if c > 0: ok_forbid=False; break
        except Exception as e:
            logging.warning(f"Neo4j check error: {e}; falling back to NX.")
            neo4j_driver=None

    # NX path
    if neo4j_driver is None and nxG is not None:
        for n in req_nodes:
            if n not in nxG.nodes:
                ok_nodes=False; break
        if ok_nodes:
            for (a, rel, b) in req_edges:
                alts = [rel] + (allow_alts.get(rel, []) if isinstance(allow_alts.get(rel, []), list) else [])
                edge_ok=False
                if nxG.has_edge(a,b):
                    et = nxG.edges[(a,b)].get("relationship", nxG.edges[(a,b)].get("rel",""))
                    if et in alts: edge_ok=True
                if not edge_ok:
                    ok_edges=False; break
        if ok_edges:
            for (a, rel, b) in forbid_edges:
                if nxG.has_edge(a,b):
                    et = nxG.edges[(a,b)].get("relationship", nxG.edges[(a,b)].get("rel",""))
                    if et == rel:
                        ok_forbid=False; break

    return {
        "program": {"require_nodes": req_nodes, "require_edges": req_edges, "allow_alternatives": allow_alts, "forbid_edges": forbid_edges},
        "verdict": bool(ok_nodes and ok_edges and ok_forbid),
        "ok_nodes": ok_nodes, "ok_edges": ok_edges, "ok_forbid": ok_forbid,
    }

# ---------- Answer + IPS ----------
def _engine_answer_text(engine, query:str):
    for m in ["query","run","execute","__call__"]:
        if hasattr(engine,m) and callable(getattr(engine,m)):
            try:
                out=getattr(engine,m)(query)
                if isinstance(out,str): return out
                if isinstance(out,dict):
                    for k in ["response","answer","text","output"]:
                        if k in out and isinstance(out[k],str): return out[k]
                    return json.dumps(out)[:1500]
                return str(out)[:1500]
            except Exception: pass
    return None

def _llm_refine(llm, text:str)->str:
    adapter = llm if isinstance(llm,_RobustLLMAdapter) else _RobustLLMAdapter(llm)
    out=adapter.call(f"Refine the following answer in a single precise sentence for the user:\n{text}")
    return out or text

def _draft_from_evidence(llm, ctx)->Tuple[str,float]:
    adapter = llm if isinstance(llm,_RobustLLMAdapter) else _RobustLLMAdapter(llm)
    bits=[e.get("text","") for e in ctx.get("evidence",[])[:3]]
    raw=" ".join(bits) or "Insufficient evidence."
    refined=adapter.call(f"Draft a one-sentence answer using ONLY this evidence:\n{raw}")
    logp = -math.log1p(max(1, 6 - len(ctx.get('evidence',[]))))
    return (refined or raw), float(logp)

def generate_answer_with_llm(llm, ctx, engine=None, query:str=""):
    adapter = llm if isinstance(llm,_RobustLLMAdapter) else _RobustLLMAdapter(llm)
    base=_engine_answer_text(engine, query) if engine is not None else None
    if base: return _llm_refine(adapter, base), -math.log1p(1.0)
    return _draft_from_evidence(adapter, ctx)

def ips_ablate_and_attribute(answer_fn, base_ctx, paths, node_layers):
    from copy import deepcopy
    attributions={"nodes":{},"edges":{}}
    _, base_lp = answer_fn(base_ctx)
    def drop(ctx2): _, lp = answer_fn(ctx2); return max(0.0, base_lp - lp)
    for p in paths:
        for nid in p:
            ctx2=deepcopy(base_ctx); ctx2["evidence"]=[e for e in ctx2["evidence"] if e.get("node_id")!=nid]
            d=drop(ctx2); meta=attributions["nodes"].setdefault(nid, {"impact":0.0,"layer":node_layers.get(nid,0)})
            meta["impact"]=max(meta["impact"], float(d))
    for p in paths:
        for a,b in zip(p,p[1:]):
            ctx2=deepcopy(base_ctx); ctx2["evidence"]=[e for e in ctx2["evidence"] if not (e.get("src")==a and e.get("tgt")==b)]
            d=drop(ctx2); key=f"{a}->{b}"
            meta=attributions["edges"].setdefault(key, {"impact":0.0,"src":a,"tgt":b,"layer_src":node_layers.get(a,0),"layer_tgt":node_layers.get(b,0)})
            meta["impact"]=max(meta["impact"], float(d))
    for nid,meta in attributions["nodes"].items():
        L=meta["layer"]; meta["layer_credit"]=meta["impact"]/(1+L)
    base_ans,_=answer_fn(base_ctx)
    return attributions,(base_ans,base_lp)

def ccr_contrastive_reports(llm, kept_comm, rejected_comm):
    js=_llm_json_strict(llm, f"""
Return ONLY JSON with keys: "kept_strengths", "rival_gaps", "minimal_snippets".
KEPT:
{json.dumps(kept_comm, default=str)}
RIVAL:
{json.dumps(rejected_comm, default=str)}
""", {"kept_strengths": [], "rival_gaps": [], "minimal_snippets": []})
    return js

# ---------- Orchestrator ----------
def run_reason_explain_on_pipeline(pipeline, query:str, json_only:bool=False)->Dict[str,Any]:
    llm_adapter=_RobustLLMAdapter(pipeline.llm)
    store=pipeline.graph_store; _ensure_store_api(store)
    nxG=store._create_nx_graph()
    neo4j_driver=_safe_neo4j_driver(store)  # probe; may return None

    hyps=hrr_propose_hypotheses(llm_adapter, query, n=3)
    pools=hrr_pool_per_hypothesis(store, query, hyps, k=20)
    per_hyp_filtered={h: layerwise_filter(store, res, threshold=0.05) for h,res in pools.items()}

    def _S(f): vals=[x["relevancescore"] for _,items in f.items() for x in items]; return float(np.mean(vals)) if vals else 0.0
    def _N(f): vals=[1.0 if x.get("is_decisive") else 0.0 for _,items in f.items() for x in items]; return float(np.mean(vals)) if vals else 0.0
    SN={h:0.7*_S(f)+0.3*_N(f) for h,f in per_hyp_filtered.items()} if per_hyp_filtered else {}
    best_h=max(SN,key=SN.get) if SN else None
    filtered_layers=per_hyp_filtered.get(best_h,{}) if best_h else {}

    if not filtered_layers or all(len(v)==0 for v in filtered_layers.values()):
        logging.warning("[Fallback] No hits. Retrying with threshold=0.0, k=50")
        pools2=hrr_pool_per_hypothesis(store, query, hyps, k=50)
        per_hyp_filtered2={h: layerwise_filter(store, res, threshold=0.0) for h,res in pools2.items()}
        SN2={h:0.7*_S(f)+0.3*_N(f) for h,f in per_hyp_filtered2.items()} if per_hyp_filtered2 else {}
        best_h2=max(SN2,key=SN2.get) if SN2 else None
        filtered_layers=per_hyp_filtered2.get(best_h2,{}) if best_h2 else filtered_layers
        if best_h2: best_h=best_h2

    if not filtered_layers or all(len(v)==0 for v in filtered_layers.values()):
        logging.warning("[Fallback] Still empty. Synthesizing minimal evidence from top communities.")
        hcomms=getattr(store,"hierarchical_communities",{}) or {}
        top_layer=max(hcomms.keys()) if hcomms else 0
        comms=(hcomms.get(top_layer) or [])[:3]
        if comms:
            filtered_layers={top_layer:[{"id":c.get("id"),"type":"node","layer":top_layer,"relevancescore":0.0,"is_decisive":False,"content":{"summary":c.get("summary","")}} for c in comms]}
            if not best_h: best_h=hyps[0] if hyps else "H1"

    cogt=cogt_reason(llm_adapter, query, filtered_layers)
    paths=infer_paths_from_cogt(cogt, filtered_layers=filtered_layers)
    rgp=rgp_emit_and_verify(llm_adapter, query, cogt_steps=cogt, nxG=nxG, neo4j_driver=neo4j_driver)

    def _flatten_evidence(filtered_layers, store):
        ev=[]
        for _,items in filtered_layers.items():
            for it in items:
                nid=it["id"]; ev.append({"node_id":nid,"text":store.get_text(nid),"layer":it["layer"]})
                for nb in store.neighbors(nid)[:2]:
                    ev.append({"src":nid,"tgt":nb,"text":f"{nid} -> {nb}","layer":max(it["layer"], store.get_layer(nb))})
        return ev

    base_ctx={"query":query,"evidence":_flatten_evidence(filtered_layers, store)}
    node_layers={e["node_id"]:e.get("layer",0) for e in base_ctx["evidence"] if "node_id" in e}

    attributions,(answer,logp)=ips_ablate_and_attribute(
        answer_fn=lambda ctx: generate_answer_with_llm(llm_adapter, ctx, engine=pipeline.query_engine, query=query),
        base_ctx=base_ctx, paths=paths, node_layers=node_layers
    )

    def _pick_top(fl):
        best=None
        for _,items in fl.items():
            for it in items:
                if (best is None) or (it["relevancescore"]>best["relevancescore"]): best=it
        return best or {}
    def _rival(fl):
        k=_pick_top(fl)
        if not k: return {}
        r=dict(k); r["id"]=k["id"]+"_rival"; r["relevancescore"]=max(0.0, k["relevancescore"]-0.05); return r

    ccr=ccr_contrastive_reports(llm_adapter, kept_comm=_pick_top(filtered_layers), rejected_comm=_rival(filtered_layers))

    out={"query":query,"hypotheses":hyps,"chosen_hypothesis":best_h,"filtered_layers":filtered_layers,
         "reasoning":{"cogt":cogt,"rgp":rgp,"paths":paths},
         "explanation":{"attributions":attributions,"contrastive":ccr},
         "answer":answer,"answer_logprob_proxy":logp,"adapter_demo_mode":False}

    if not json_only:
        print("=== Reason+Explain on ArchRAG ===")
        print("Query:", query); print("Chosen Hypothesis:", best_h)
        print("\n-- CoGT --")
        for i,s in enumerate(out["reasoning"]["cogt"]):
            if isinstance(s,dict):
                print(f"  {i+1}. [{s.get('node_id')}] {s.get('claim')} (conf={s.get('confidence')})")
        print("\n-- RGP Verdict --"); print(json.dumps(out["reasoning"]["rgp"], indent=2))
        print("\n-- Answer --"); print(out["answer"])
        print("\n-- Top Node Attributions --")
        nodes_sorted=sorted(out["explanation"]["attributions"]["nodes"].items(), key=lambda kv:-kv[1]["impact"])[:5]
        for nid,meta in nodes_sorted:
            print(f"  {nid}: impact={meta['impact']:.3f}, layer={meta['layer']}, layer_credit={meta['layer_credit']:.3f}")
    return out

# ---------- Convenience wrappers (reuse existing pipeline/graph) ----------
def reason_explain_only(pipeline, query, json_only=False):
    if not hasattr(pipeline,"llm") or pipeline.llm is None:
        pipeline.llm = build_hf_chat_llm()
    if not hasattr(pipeline,"graph_store") or pipeline.graph_store is None:
        raise RuntimeError("pipeline.graph_store is missing.")
    return run_reason_explain_on_pipeline(pipeline, query, json_only=json_only)

def _archrag_diagnostics_run(pipeline, example_query="What connects insulin resistance to obesity?"):
    # 1) LLM ready
    try:
        if not hasattr(pipeline,"llm") or pipeline.llm is None:
            pipeline.llm = build_hf_chat_llm("mistralai/Mistral-7B-Instruct-v0.3", prefer_4bit=True)
        print("[OK] LLM attached:", type(pipeline.llm))
    except Exception as e:
        print("[ERR] LLM attach failed:", e)

    # 2) Graph summary (skip if Neo4j down)
    try:
        if not hasattr(pipeline,"graph_store") or pipeline.graph_store is None:
            raise RuntimeError("pipeline.graph_store is missing.")
        G = pipeline.graph_store._create_nx_graph()
        print(f"[INFO] Graph nodes={len(G.nodes)}, edges={len(G.edges)}")
        ents = getattr(pipeline.graph_store,"entity_embeddings",{})
        print(f"[INFO] entity_embeddings={len(ents)} entries")
    except Exception as e:
        print("[ERR] Graph check failed:", e)

    # 3) LLM smoke test
    try:
        raw=_RobustLLMAdapter(pipeline.llm).call('Return only JSON: ["ok"]')
        print("[INFO] LLM smoke test output:", raw)
    except Exception as e:
        print("[ERR] LLM smoke test failed:", e)

    # 4) One visible run
    try:
        print(f"\n[RUN] {example_query}")
        out=run_reason_explain_on_pipeline(pipeline, example_query, json_only=False)
        print("\n[RESULT] keys:", list(out.keys()))
        fl=out.get("filtered_layers", {})
        print(f"[SUMMARY] hypotheses={out.get('hypotheses')}, chosen={out.get('chosen_hypothesis')}, hits={sum(len(v) for v in fl.values())}")
        return out
    except Exception as e:
        import traceback
        print("[ERR] run failed:", e); print(traceback.format_exc()); return {}

# ---------- Use it (reuses your existing ArchRAGPipeline/config) ----------
try:
    pipeline  # reuse if already constructed above
except NameError:
    pipeline = ArchRAGPipeline(**config)
    pipeline.build_knowledge_graph(max_docs=1)
    
    G = pipeline.graph_store._create_nx_graph()
    print(f"[CHECK] Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")
    for nid in G.nodes:
        text = pipeline.graph_store.get_text(nid)
        if text and len(text.split()) > 3:  # filter meaningful ones
            print(f"[DEBUG] Node {nid} -> Text to embed: {text}")
            break

    # After building KG
    #pipeline.embed_entities()
    pipeline.embed_entities(layer_range=(0, 3))  # or whatever top layers are relevant
    print("[CHECK] Embedded entities:", len(pipeline.graph_store.entity_embeddings))


# Optional: Print sample node to verify embeddings
    print("Sample embedding:", next(iter(pipeline.graph_store.entity_embeddings.items())))


# 1) Quick diagnostics on current LLM
_ = _archrag_diagnostics_run(pipeline)

# 2) Swap to Llama 3.1 8B with safe backoffs (no rebuild)
pipeline.llm = build_hf_chat_llm("meta-llama/Meta-Llama-3.1-8B-Instruct", prefer_4bit=True)

# 3) Run Reason+Explain only (no rebuild)
out = reason_explain_only(pipeline, "What connects insulin resistance to obesity?", json_only=False)


In [ ]:
# ============================
# Reason+Explain (Batch-enabled, CSV output, CoGT viz, community fallback)
# ============================

import os, json, math, re, logging
from typing import Any, Dict, List, Tuple
import pandas as pd
import networkx as nx
from pathlib import Path
import matplotlib.pyplot as plt

# Your existing imports for transformers, langchain, and numpy remain
# (We assume they are loaded from previous cells as per your instructions)

# ---- CSV-safe flatten helper ----
def _flatten_results_for_csv(query: str, out: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "Query": query,
        "ChosenHypothesis": out.get("chosen_hypothesis"),
        "Answer": out.get("answer"),
        "AnswerLogProb": out.get("answer_logprob_proxy"),
        "RGP_Verdict": out.get("reasoning", {}).get("rgp", {}).get("verdict"),
        "RGP_Program": json.dumps(out.get("reasoning", {}).get("rgp", {}).get("program", {})),
        "Paths": json.dumps(out.get("reasoning", {}).get("paths")),
        "TopAttributions": json.dumps(out.get("explanation", {}).get("attributions", {}).get("nodes", {})),
        "ContrastiveGaps": json.dumps(out.get("explanation", {}).get("contrastive", {}).get("rival_gaps", [])),
    }

# ---- Visualize CoGT with Communities ----
def visualize_cogt_communities(filtered_layers: Dict[int, List[Dict]], paths: List[List[str]], chosen_hyp: str):
    G = nx.DiGraph()
    pos = {}
    colors = []
    labels = {}

    # Place nodes by layer (y) and index (x)
    for y, (layer, items) in enumerate(sorted(filtered_layers.items(), reverse=True)):
        for x, node in enumerate(items):
            nid = node['id']
            G.add_node(nid)
            pos[nid] = (x, -y)
            labels[nid] = nid
            if node.get("is_decisive"):
                colors.append("green" if chosen_hyp in nid else "lightblue")
            else:
                colors.append("gray")

    # Edges from paths
    for path in paths:
        for a, b in zip(path, path[1:]):
            if a in G and b in G:
                G.add_edge(a, b)

    plt.figure(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, labels=labels, node_color=colors, node_size=1000, font_size=8, arrows=True)
    plt.title(f"CoGT Graph for Hypothesis: {chosen_hyp}")
    plt.show()

# ---- Batch Processor ----
def run_batch_reason_explain(pipeline, queries_path: Path, csv_path: Path = None, visualize: bool = True):
    queries = []
    if queries_path.suffix == ".jsonl":
        with open(queries_path) as f:
            for line in f:
                queries.append(json.loads(line.strip()))
    elif queries_path.suffix == ".json":
        with open(queries_path) as f:
            raw = json.load(f)
            if isinstance(raw, list):
                queries.extend(raw)
            elif isinstance(raw, dict):
                queries.append(raw)
    else:
        raise ValueError("Unsupported format. Use .json or .jsonl")

    results = []
    for i, item in enumerate(queries):
        query = item.get("question") or item.get("query")
        if not query:
            continue
        print(f"[{i+1}/{len(queries)}] {query}")
        try:
            out = reason_explain_only(pipeline, query, json_only=True)
            if visualize:
                visualize_cogt_communities(
                    out.get("filtered_layers", {}),
                    out.get("reasoning", {}).get("paths", []),
                    out.get("chosen_hypothesis", "")
                )
            results.append(_flatten_results_for_csv(query, out))
        except Exception as e:
            print(f"[ERR] Failed: {e}")
            results.append({"Query": query, "Error": str(e)})

    df = pd.DataFrame(results)
    if csv_path:
        df.to_csv(csv_path, index=False)
        print(f"✅ Written to {csv_path}")
    return df

# ---- Run it (single call) ----
DATASET_PATH = Path(os.getenv("HAGRAG_DATASET_PATH", "data/100diabetes_qa_dataset.jsonl"))
CSV_OUTPUT_PATH = OUTPUT_DIR / "results.csv"
batch_df = run_batch_reason_explain(pipeline, DATASET_PATH, csv_path=CSV_OUTPUT_PATH)


In [ ]:
# --- Test query for Diabetes Prevention Program ---
test_query = (
    "What is insulin resistance?,"
    "What is diabetes?"
)

# Run Reason+Explain with verbose output
out = reason_explain_only(pipeline, test_query, json_only=False)

# Fallback: Synthesize CoGT steps if none exist
if not out["reasoning"]["cogt"]:
    top_items = []
    for layer, items in sorted(out.get("filtered_layers", {}).items(), reverse=True):
        top_items.extend(items)
        if len(top_items) >= 3:
            break
    cogt_fallback = []
    for i, node in enumerate(top_items[:3]):
        cogt_fallback.append({
            "node_id": node["id"],
            "node_type": "Community",
            "claim": f"Evidence from node: {node['content']['summary'][:60]}",
            "supporting_snippets": [node["content"]["summary"]],
            "confidence": 0.3,
            "expected_counter_evidence": ""
        })
    out["reasoning"]["cogt"] = cogt_fallback
    print("\n[Fallback] Synthesized CoGT steps from top evidence nodes.")

# --- Display CoGT steps (textual) ---
print("\n--- CoGT Reasoning Steps ---")
for i, step in enumerate(out["reasoning"]["cogt"]):
    print(f"{i+1}. [{step.get('node_id')}] {step.get('claim')} (conf={step.get('confidence')})")

# --- Optional: Draw a graph ---
import matplotlib.pyplot as plt
import networkx as nx

def draw_cogt_graph(cogt_steps):
    if not cogt_steps:
        print("No CoGT to display.")
        return
    G = nx.DiGraph()
    for i, step in enumerate(cogt_steps):
        nid = step.get("node_id", f"Step {i+1}")
        label = f"{nid}\n{step.get('claim', '')[:40]}"
        G.add_node(nid, label=label)
        if i > 0:
            prev_nid = cogt_steps[i - 1].get("node_id", f"Step {i}")
            G.add_edge(prev_nid, nid)
    pos = nx.spring_layout(G)
    plt.figure(figsize=(12, 6))
    nx.draw(G, pos, with_labels=True, labels=nx.get_node_attributes(G, 'label'),
            node_color="lightblue", node_size=1600, font_size=9, edge_color="gray")
    plt.title("CoGT Reasoning Graph")
    plt.show()

# Visualize the reasoning chain
draw_cogt_graph(out["reasoning"]["cogt"])
